# Two-Layer FD Solver — `Solver` class

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.interpolate import CubicSpline

torch.set_default_dtype(torch.float64)

In [2]:
%run func_defs.ipynb

In [3]:
def set_BCs(T):
    T[0]  = T[2]
    T[-1] = T[-2] ** 2 / max(T[-3].item(), 1e-40)

In [4]:
def solve_two_layer_FD(k1, k2, z_dis, Pda, Lz, nz, tf, n_t_out, device, verbose=True):
    """
    Solve the two-layer heat equation from t=0 to t=tf.

    Marches at the CFL-stable dt; outputs are resampled to
    n_t_out equally-spaced coarse times via cubic spline.

    Returns
    -------
    z        : (nz+1,)           spatial grid
    t_out    : (n_t_out,)        coarse output times
    U_coarse : (nz+1, n_t_out)  temperature field
    """
    dz = Lz / nz
    z  = torch.arange(-1, nz + 2, device=device) * dz   # (nz+3,)

    m = torch.searchsorted(z, z_dis - 0.5 * dz, right=True).item()
    if verbose:
        print(f'Interface alignment error: {z[m].item() - z_dis:.2e}')

    k_iL = 8 * k1 * k2 / (k1 + 7 * k2)
    k_iR = 8 * k1 * k2 / (7 * k1 + k2)
    kh = torch.where(
        z[:-1] < z_dis - 0.5 * dz,
        torch.tensor(k1, device=device),
        torch.tensor(k2, device=device),
    )  # (nz+2,)
    kh[m - 1] = k_iL
    kh[m]     = k_iR

    dt_stab = 0.5 * (dz ** 2) / max(k1, k2)
    n_fine  = int(np.ceil(tf / dt_stab))
    dt      = tf / n_fine
    if verbose:
        print(f'n_fine={n_fine}, dt={dt:.4e}, tf={tf}')

    source    = Pda * torch.exp(-z[1:-1])
    source[0] = Pda * torch.exp(-z[2] / 4)

    T  = torch.zeros(nz + 3, device=device)
    Jh = torch.zeros(nz + 2, device=device)

    stride        = max(1, n_fine // (10 * n_t_out))
    buf_steps     = list(range(stride - 1, n_fine, stride))
    if (n_fine - 1) not in buf_steps:
        buf_steps.append(n_fine - 1)
    buf_steps_set = set(buf_steps)

    buf_U = []
    buf_t = []

    for step in range(n_fine):
        set_BCs(T)
        Jh = -kh * (T[1:] - T[:-1]) / dz
        T[1:-1] = T[1:-1] + dt / dz * (Jh[:-1] - Jh[1:]) + dt * source
        if step in buf_steps_set:
            buf_U.append(T[1:-1].cpu().clone())
            buf_t.append((step + 1) * dt)

    set_BCs(T)
    T[m] = T[m] + (2 * T[m] - T[m - 1] - T[m + 1]) / 6

    t_fine_buf = np.array(buf_t)
    U_fine_buf = torch.stack(buf_U, dim=1).numpy()           # (nz+1, n_buf)

    t_coarse = np.linspace(tf / n_t_out, tf, n_t_out)
    cs       = CubicSpline(t_fine_buf, U_fine_buf, axis=1, bc_type='not-a-knot')
    U_coarse = torch.from_numpy(cs(t_coarse)).to(device)     # (nz+1, n_t_out)

    t_out = torch.tensor(t_coarse, device=device)
    return z[1:-1], t_out, U_coarse

In [5]:
def smooth_U(U, z, z_dis, k1, k2):
    """
    Smooth the kink at z_dis:
        z >= z_dis : unchanged
        z <  z_dis : U(z_dis, t) + (k1/k2) * (U(z,t) - U(z_dis, t))
    """
    alpha    = k1 / k2
    m        = torch.searchsorted(z, torch.tensor(z_dis, device=z.device)).item()
    U_smooth = U.clone()
    U_zd     = U[m, :]
    U_smooth[:m, :] = U_zd + alpha * (U[:m, :] - U_zd)
    return U_smooth

In [6]:
_VALID_KEYS = frozenset({
    'U', 'U_raw', 'z', 't', 'stau', 'W', 'kratio', 'z_dis', 'dataset',
})


class Solver:
    """
    Two-layer FD solver with dictionary-style output access.

    Parameters
    ----------
    k1, k2    : thermal diffusivities (layer 1, layer 2)
    z_dis     : interface depth
    Pda       : source amplitude
    Lz        : domain depth
    nz        : number of spatial cells
    tf        : final time
    n_t_out   : number of coarse output time points
    device    : 'cpu' or 'cuda'
    verbose   : print solver diagnostics (default True)

    Dict keys
    ---------
    'U'       (nz+1, n_t_out)         smoothed temperature
    'U_raw'   (nz+1, n_t_out)         raw temperature (no smoothing)
    'z'       (nz+1,)                 spatial grid
    't'       (n_t_out,)              coarse output times
    'stau'    ((nz+1)*n_t_out, 2)     (s, tau) coordinates
    'W'       ((nz+1)*n_t_out, 1)     W = U_smooth / t
    'kratio'  flat tensor of k1/k2
    'z_dis'   flat tensor of interface location
    'dataset' dict ready for torch.save
    """

    def __init__(self, k1, k2, z_dis, Pda, Lz, nz, tf, n_t_out,
                 device='cpu', verbose=True):
        self._p = dict(k1=k1, k2=k2, z_dis=z_dis, Pda=Pda,
                       Lz=Lz, nz=nz, tf=tf, n_t_out=n_t_out,
                       device=device)
        self._verbose = verbose
        self._cache   = {}
        self._run()

    # ------------------------------------------------------------------
    def _run(self):
        p = self._p

        # 1. solve PDE
        z, t_out, U_raw = solve_two_layer_FD(
            p['k1'], p['k2'], p['z_dis'], p['Pda'],
            p['Lz'], p['nz'], p['tf'], p['n_t_out'],
            p['device'], verbose=self._verbose,
        )

        # 2. smooth across the interface
        U_smooth = smooth_U(U_raw, z, p['z_dis'], p['k1'], p['k2'])

        # 3. (s, tau) coordinates — uses z_t_to_s_tau from func_defs.ipynb
        zt_grid      = torch.cartesian_prod(z, t_out)
        zz, tt       = zt_grid[:, 0], zt_grid[:, 1]
        ss, ttau     = z_t_to_s_tau(zz, tt)
        stau         = torch.column_stack((ss, ttau))

        # 4. scaled solution  W = U_smooth / t
        W = (U_smooth / t_out).reshape(-1, 1)

        # 5. per-point parameter tensors
        n_pts  = len(stau)
        kratio = torch.full((n_pts,), p['k1'] / p['k2'], device=p['device'])
        z_disv = torch.full((n_pts,), p['z_dis'],        device=p['device'])

        self._cache = {
            'U':      U_smooth,
            'U_raw':  U_raw,
            'z':      z,
            't':      t_out,
            'stau':   stau,
            'W':      W,
            'kratio': kratio,
            'z_dis':  z_disv,
        }

    # ------------------------------------------------------------------
    def __getitem__(self, key):
        if key not in _VALID_KEYS:
            raise KeyError(f'Unknown key {key!r}. Valid keys: {sorted(_VALID_KEYS)}')
        if key == 'dataset':
            return self._build_dataset()
        return self._cache[key]

    def _build_dataset(self):
        c = self._cache
        return {
            't':      c['stau'][:, 1],
            'z':      c['stau'][:, 0],
            'kratio': c['kratio'],
            'zdis':   c['z_dis'],
            'stau':   c['stau'],
            'W':      c['W'],
        }

    def keys(self):
        return sorted(_VALID_KEYS)

    def save(self, path):
        """Save the dataset dict to *path* via torch.save."""
        torch.save(self._build_dataset(), path)
        if self._verbose:
            print(f'Dataset saved to {path}')

    def __repr__(self):
        p = self._p
        return (f"Solver(k1={p['k1']}, k2={p['k2']}, z_dis={p['z_dis']}, "
                f"Pda={p['Pda']}, Lz={p['Lz']}, nz={p['nz']}, "
                f"tf={p['tf']}, n_t_out={p['n_t_out']}, device={p['device']!r})")

## Run the solver

Edit parameters below, then run.

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
Lz = 8.0
sol = Solver(
    k1      = 1.0,
    k2      = 2.0,
    z_dis   = 1.0,
    Pda     = 1.0,
    Lz      = 8.0,
    nz      = int(Lz) * 10,   # nz_unit = 10  →  nz = 400
    tf      = 2.0,
    n_t_out = 400,
    device  = device,
)

print(sol)
print('Available keys:', sol.keys())

Interface alignment error: 0.00e+00
n_fine=800, dt=2.5000e-03, tf=2.0
Solver(k1=1.0, k2=2.0, z_dis=1.0, Pda=1.0, Lz=8.0, nz=80, tf=2.0, n_t_out=400, device='cuda')
Available keys: ['U', 'U_raw', 'W', 'dataset', 'kratio', 'stau', 't', 'z', 'z_dis']


In [13]:
U      = sol['U']       # (nz+1, n_t_out)  smoothed temperature
U_raw  = sol['U_raw']   # (nz+1, n_t_out)  raw temperature
z      = sol['z']       # (nz+1,)
t      = sol['t']       # (n_t_out,)
stau   = sol['stau']    # ((nz+1)*n_t_out, 2)
W      = sol['W']       # ((nz+1)*n_t_out, 1)
kratio = sol['kratio']
z_dis  = sol['z_dis']

print(f'U      : {U.shape}')
print(f'U_raw  : {U_raw.shape}')
print(f'z      : {z.shape}')
print(f't      : {t.shape}')
print(f'stau   : {stau.shape}')
print(f'W      : {W.shape}')

U      : torch.Size([81, 400])
U_raw  : torch.Size([81, 400])
z      : torch.Size([81])
t      : torch.Size([400])
stau   : torch.Size([32400, 2])
W      : torch.Size([32400, 1])


## Quick plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

z_cpu  = z.cpu();   t_cpu  = t.cpu()
U_cpu  = U.cpu();   Ur_cpu = U_raw.cpu()
idx    = torch.linspace(0, U_cpu.shape[1] - 1, 6).long()
m      = torch.searchsorted(z_cpu, torch.tensor(sol._p['z_dis'])).item()
nz_    = z.shape[0] - 1

# smoothed
ax = axes[0]
for i in idx:
    ax.plot(z_cpu.numpy(), U_cpu[:, i].numpy(), label=f't={t_cpu[i]:.3f}')
ax.axvline(sol._p['z_dis'], color='k', linestyle='--', linewidth=0.8, label='z_dis')
ax.set_xlabel('z'); ax.set_ylabel('U')
ax.set_title('U(z,t) — smoothed'); ax.legend(fontsize=7)

# raw
ax = axes[1]
for i in idx:
    ax.plot(z_cpu.numpy(), Ur_cpu[:, i].numpy(), label=f't={t_cpu[i]:.3f}')
ax.axvline(sol._p['z_dis'], color='k', linestyle='--', linewidth=0.8, label='z_dis')
ax.set_xlabel('z'); ax.set_ylabel('U')
ax.set_title('U(z,t) — raw'); ax.legend(fontsize=7)

# zoom near interface
ax = axes[2]
i_last = U_cpu.shape[1] - 1
w = max(1, nz_ // 10)
ax.plot(z_cpu[m-w:m+w].numpy(), U_cpu[m-w:m+w, i_last].numpy(),  'b-',  label='smoothed')
ax.plot(z_cpu[m-w:m+w].numpy(), Ur_cpu[m-w:m+w, i_last].numpy(), 'r--', label='raw')
ax.axvline(sol._p['z_dis'], color='k', linestyle='--', linewidth=0.8)
ax.set_xlabel('z'); ax.set_ylabel('U')
ax.set_title('Zoom near z_dis (last snapshot)'); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Save dataset

In [ ]:
p        = sol._p
out_file = f'./data/2layer_k1{p["k1"]}_k2{p["k2"]}_nz{p["nz"]}.pt'
sol.save(out_file)